# Application Data – Feature Engineering

Step 1 of preprocessing for the main application table: one row per applicant
with the details they gave when applying (income, loan amount, age, job,
family, housing) and the `TARGET` column (1 = had payment difficulties).

Reads `data/Input/application_train.csv`, cleans placeholder values, adds new
features, and saves the result for the next steps.

**Sections**
1. Settings
2. Load the raw data
3. Turn placeholder values (`365243`, `XNA`) into missing
4. Financial ratios, age/employment in years, and the external-score average
5. Document and contact counts
6. How recently the phone number changed
7. Building/housing indices
8. Yes/no property and building flags
9. Drop unhelpful columns
10. Quality checks
11. Save

**Output:** `data/Model-Training/Staging/application_train_data_processed.parquet`

Run by the `01_preprocessing` Airflow DAG (task `task_application`), or by hand.

## 1. Imports & Configuration

`DATA_ROOT` is the project's `data/` folder: `/data` inside Docker (set by
docker-compose), otherwise this project's folder on your PC.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_APP_TRAIN = os.path.join(DATA_ROOT, "Input", "application_train.csv")                                     # raw input
PATH_OUTPUT    = os.path.join(DATA_ROOT, "Model-Training", "Staging", "application_train_data_processed.parquet")  # result

## 2. Load Data

`SK_ID_CURR` (the applicant ID) is read as text so it joins cleanly with the other tables later.

In [2]:
application_train = pd.read_csv(PATH_APP_TRAIN)
application_train = application_train.astype({'SK_ID_CURR': 'string'})

print(f"Shape: {application_train.shape}")

Shape: (307511, 122)


## 3. Turn Placeholder Values into Missing

### 3a. The number `365243`

The data uses `365243` days (about 1,000 years) to mean "not applicable" —
for example `DAYS_EMPLOYED` for pensioners and the unemployed. Left in, it
would look like a huge real value. Every numeric column is checked, not
just `DAYS_EMPLOYED`, and every `365243` becomes missing (`NaN`).

In [3]:
SENTINEL = 365243
numeric_cols = application_train.select_dtypes(include='number').columns
sentinel_cols = [
    col for col in numeric_cols
    if (application_train[col] == SENTINEL).any()
]
print(f"Columns containing sentinel {SENTINEL}: {sentinel_cols}")

for col in sentinel_cols:
    application_train.loc[application_train[col] == SENTINEL, col] = np.nan

print("Sentinel values replaced with NaN.")

Columns containing sentinel 365243: ['DAYS_EMPLOYED']
Sentinel values replaced with NaN.


### 3b. The text `'XNA'`

`XNA` ("not available") marks unknown values in text columns, e.g.
`ORGANIZATION_TYPE` and `CODE_GENDER`. Every text column is checked and
`XNA` becomes missing, so it isn't treated as a real category later.

In [4]:
object_cols = application_train.select_dtypes(include='object').columns
xna_cols = [
    col for col in object_cols
    if (application_train[col] == 'XNA').any()
]
print(f"Columns containing 'XNA': {xna_cols}")

for col in xna_cols:
    application_train.loc[application_train[col] == 'XNA', col] = np.nan

print("XNA values replaced with NaN.")

Columns containing 'XNA': ['CODE_GENDER', 'ORGANIZATION_TYPE']
XNA values replaced with NaN.


## 4. Financial Ratios, Age/Employment & External Scores

- **`GOODSPRICE_TO_AMT_CREDIT`** — price of the goods ÷ loan amount. Below 1 means borrowing more than the goods cost.
- **`AMT_ANNUITY_TO_CREDIT_RATIO`** — yearly repayment ÷ loan amount.
- **`AMT_ANNUITY_TO_INCOME_RATIO`** — yearly repayment ÷ income: how much of their income goes on this loan.
- **`INCOME_PER_FAMILY`** — income ÷ number of family members.
- **`AGE`** — age in years, from `DAYS_BIRTH`.
- **`EXT_SCORE_MEAN`** — the average of the three external credit scores
  (`EXT_SOURCE_1/2/3`, each 0–1, higher = safer), using whichever are
  present; blank only when all three are missing. It's added alongside the
  three, which are kept. (`EXT_SOURCE_1` is missing for over half of
  applicants, so the average gives everyone with at least one score a
  single, comparable value.)

The `DAYS_*` columns count days before the application as negative numbers,
so dividing by −365 gives positive years. Each ratio is left blank when its
denominator is 0 or missing.

`DAYS_EMPLOYED` (days in the current job; blank for pensioners and the
unemployed after step 3a) is kept as it is rather than also converted to
years: for tree models a rescaled copy adds no information, and the @prod
model uses `DAYS_EMPLOYED`.

In [5]:
# Each ratio is left blank (NaN) when the denominator is 0 or missing
application_train['GOODSPRICE_TO_AMT_CREDIT'] = np.where(
    application_train['AMT_CREDIT'] > 0,
    application_train['AMT_GOODS_PRICE'] / application_train['AMT_CREDIT'],
    np.nan,
)
application_train['AMT_ANNUITY_TO_CREDIT_RATIO'] = np.where(
    application_train['AMT_CREDIT'] > 0,
    application_train['AMT_ANNUITY'] / application_train['AMT_CREDIT'],
    np.nan,
)
application_train['AMT_ANNUITY_TO_INCOME_RATIO'] = np.where(
    application_train['AMT_INCOME_TOTAL'] > 0,
    application_train['AMT_ANNUITY'] / application_train['AMT_INCOME_TOTAL'],
    np.nan,
)
application_train['INCOME_PER_FAMILY'] = np.where(
    application_train['CNT_FAM_MEMBERS'] > 0,
    application_train['AMT_INCOME_TOTAL'] / application_train['CNT_FAM_MEMBERS'],
    np.nan,
)

# Days before application (negative) -> positive years
application_train['AGE'] = application_train['DAYS_BIRTH'] / -365

# Average of the three external credit scores. mean() skips missing values,
# so it uses whichever scores are present; blank only if all three are missing.
ext_cols = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']
application_train['EXT_SCORE_MEAN'] = application_train[ext_cols].mean(axis=1)

# Not used: other summaries of the external scores
# application_train['EXT_SOURCES_MIN']  = application_train[ext_cols].min(axis=1)
# application_train['EXT_SOURCES_MAX']  = application_train[ext_cols].max(axis=1)

print("Financial & demographic features added.")

Financial & demographic features added.


C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\59588898.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['GOODSPRICE_TO_AMT_CREDIT'] = np.where(
C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\59588898.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['AMT_ANNUITY_TO_CREDIT_RATIO'] = np.where(
C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\59588898.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has

## 5. Document & Contact Counts

- **`TOTAL_DOCUMENTS_SUBMITTED`** — how many supporting documents were provided (adds up the 20 `FLAG_DOCUMENT_*` yes/no columns).
- **`CONTACT_REACHABILITY_SCORE`** — how many of 5 contact methods were given (mobile, work phone, etc.), 0 to 5.

In [6]:
doc_cols = [col for col in application_train.columns if col.startswith('FLAG_DOCUMENT_')]
application_train['TOTAL_DOCUMENTS_SUBMITTED'] = application_train[doc_cols].sum(axis=1)

contact_flags = ['FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL']
application_train['CONTACT_REACHABILITY_SCORE'] = application_train[contact_flags].sum(axis=1)

print(f"Document columns summed: {len(doc_cols)}")
print(f"Contact score range: [{application_train['CONTACT_REACHABILITY_SCORE'].min()}, "
      f"{application_train['CONTACT_REACHABILITY_SCORE'].max()}]")

Document columns summed: 20
Contact score range: [1, 5]


C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\2192394892.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['TOTAL_DOCUMENTS_SUBMITTED'] = application_train[doc_cols].sum(axis=1)
C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\2192394892.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['CONTACT_REACHABILITY_SCORE'] = application_train[contact_flags].sum(axis=1)


## 6. How Recently the Phone Number Changed

`DAYS_LAST_PHONE_CHANGE` is days before the application (negative).

- **`YEARS_SINCE_PHONE_CHANGE`** — years since the phone number last changed.
- **`RECENT_PHONE_CHANGE_30D`** — 1 if it changed in the 30 days before applying, otherwise 0 (missing counts as 0).

In [7]:
application_train['YEARS_SINCE_PHONE_CHANGE'] = (
    application_train['DAYS_LAST_PHONE_CHANGE'] / -365
)
# Changed within the last 30 days -> 1 (a missing value compares as False -> 0)
application_train['RECENT_PHONE_CHANGE_30D'] = (
    application_train['DAYS_LAST_PHONE_CHANGE'] >= -30
).astype(int)

print("Phone-change features added.")

Phone-change features added.


C:\Users\domjo\AppData\Local\Temp\ipykernel_27760\226124504.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['YEARS_SINCE_PHONE_CHANGE'] = (
C:\Users\domjo\AppData\Local\Temp\ipykernel_27760\226124504.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['RECENT_PHONE_CHANGE_30D'] = (


## 7. Building/Housing Indices

The data describes the applicant's building with 14 measurements (number
of apartments, floors, living area, age of the building, …), each already
scaled to 0–1 and given in three versions: `_AVG`, `_MEDI` (median) and
`_MODE` (most common value).

Each index below is the average of the 14 measurements for one version.
Missing measurements are skipped, so an applicant with partial building
data still gets an index; one with none at all gets a blank.

In [7]:
housing_avg_cols = [
    'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG',
    'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG', 'NONLIVINGAPARTMENTS_AVG', 'NONLIVINGAREA_AVG',
]
housing_medi_cols = [
    'APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI',
    'YEARS_BUILD_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI',
    'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',
]
housing_mode_cols = [
    'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE', 'COMMONAREA_MODE', 'ELEVATORS_MODE', 'ENTRANCES_MODE',
    'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE',
    'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE',
]

# Average of the 14 measurements per applicant (missing ones skipped)
application_train['TOTAL_HOUSING_AREA_INDEX_AVG']    = application_train[housing_avg_cols].mean(axis=1)
application_train['TOTAL_HOUSING_AREA_INDEX_MEDIAN'] = application_train[housing_medi_cols].mean(axis=1)
application_train['TOTAL_HOUSING_AREA_INDEX_MODE']   = application_train[housing_mode_cols].mean(axis=1)

print("Housing indices added.")

Housing indices added.


C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\1462390701.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['TOTAL_HOUSING_AREA_INDEX_AVG']    = application_train[housing_avg_cols].mean(axis=1)
C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\1462390701.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['TOTAL_HOUSING_AREA_INDEX_MEDIAN'] = application_train[housing_medi_cols].mean(axis=1)
C:\Users\domjo\AppData\Local\Temp\ipykernel_40200\1462390701.py:23: PerformanceWarning: DataFrame

## 8. Yes/No Property & Building Flags

Each is 1 (yes) or 0 (no). A missing value is counted as 0 throughout — a
deliberate assumption that "unknown" means "doesn't have it".

- **`FLAG_OWN_CAR` / `FLAG_OWN_REALTY`** — owns a car / owns property (from `Y`/`N`).
- **`IS_EMERGENCY_STATE`** — the building is recorded as being in an emergency state.
- **`HAS_ELEVATOR`** — `ELEVATORS_AVG` is above 0.
- **`IS_HIGH_RISE`** — `FLOORSMAX_AVG` (scaled 0–1) is above 0.2.
- **`PREMIUM_AMENITIES_COUNT`** — how many of the last three are 1 (0 to 3).

In [9]:
# Y/N -> 1/0; unknown ownership counted as 0
application_train['FLAG_OWN_CAR']    = application_train['FLAG_OWN_CAR'].map({'Y': 1, 'N': 0}).fillna(0)
application_train['FLAG_OWN_REALTY'] = application_train['FLAG_OWN_REALTY'].map({'Y': 1, 'N': 0}).fillna(0)

# Emergency state: no data -> 0 (building assumed fine)
application_train['IS_EMERGENCY_STATE'] = (
    application_train['EMERGENCYSTATE_MODE'].map({'Yes': 1, 'No': 0}).fillna(0)
)

# Has an elevator (a missing value compares as False -> 0)
application_train['HAS_ELEVATOR'] = (
    application_train['ELEVATORS_AVG'] > 0
).astype(int)

# High-rise building: scaled max floors above 0.2
application_train['IS_HIGH_RISE'] = (
    application_train['FLOORSMAX_AVG'] > 0.2
).astype(int)

amenity_cols = ['IS_EMERGENCY_STATE', 'HAS_ELEVATOR', 'IS_HIGH_RISE']
application_train['PREMIUM_AMENITIES_COUNT'] = application_train[amenity_cols].sum(axis=1)

print("Property & amenity flags added.")

Property & amenity flags added.


C:\Users\domjo\AppData\Local\Temp\ipykernel_27760\2409780273.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['IS_EMERGENCY_STATE'] = (
C:\Users\domjo\AppData\Local\Temp\ipykernel_27760\2409780273.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  application_train['HAS_ELEVATOR'] = (
C:\Users\domjo\AppData\Local\Temp\ipykernel_27760\2409780273.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

## 9. Drop Unhelpful Columns

Removes columns with almost no predictive value, and the raw `DAYS_*`
columns that have been turned into years above.

`DAYS_EMPLOYED` is deliberately kept: the current @prod model uses it.

In [ ]:
application_train.drop(columns=['WEEKDAY_APPR_PROCESS_START', 'DAYS_LAST_PHONE_CHANGE', 'WALLSMATERIAL_MODE', 'FONDKAPREMONT_MODE', 'NAME_TYPE_SUITE', 'DAYS_BIRTH', 'HOUR_APPR_PROCESS_START'], inplace=True)

## 10. Quality Checks

Confirms no `365243` or `XNA` placeholders are left, and shows the share
of missing values in each new feature.

In [10]:
# Confirm sentinel 365243 is fully gone
remaining_sentinel = [
    col for col in application_train.select_dtypes(include='number').columns
    if (application_train[col] == 365243).any()
]
assert not remaining_sentinel, f"Sentinel still present in: {remaining_sentinel}"
print("No sentinel 365243 values remaining.")

# Confirm XNA is fully gone
remaining_xna = [
    col for col in application_train.select_dtypes(include='object').columns
    if (application_train[col] == 'XNA').any()
]
assert not remaining_xna, f"XNA still present in: {remaining_xna}"
print("No XNA values remaining.")

# Null rates for new features
new_features = [
    'GOODSPRICE_TO_AMT_CREDIT', 'AMT_ANNUITY_TO_CREDIT_RATIO',
    'AMT_ANNUITY_TO_INCOME_RATIO', 'INCOME_PER_FAMILY',
    'AGE', 'EXT_SCORE_MEAN',
#    'EXT_SOURCES_MIN', 'EXT_SOURCES_MAX',
    'TOTAL_DOCUMENTS_SUBMITTED', 'CONTACT_REACHABILITY_SCORE',
    'YEARS_SINCE_PHONE_CHANGE', 'RECENT_PHONE_CHANGE_30D',
    'TOTAL_HOUSING_AREA_INDEX_AVG', 'TOTAL_HOUSING_AREA_INDEX_MEDIAN',
    'TOTAL_HOUSING_AREA_INDEX_MODE',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
    'IS_EMERGENCY_STATE', 'HAS_ELEVATOR', 'IS_HIGH_RISE',
    'PREMIUM_AMENITIES_COUNT',
]
null_rates = application_train[new_features].isna().mean().sort_values(ascending=False)
print("\nNull rates for engineered features:")
print(null_rates[null_rates > 0].to_string())

print(f"\nFinal shape: {application_train.shape}")

No sentinel 365243 values remaining.
No XNA values remaining.

Null rates for engineered features:
TOTAL_HOUSING_AREA_INDEX_MODE      0.483918
TOTAL_HOUSING_AREA_INDEX_MEDIAN    0.483918
TOTAL_HOUSING_AREA_INDEX_AVG       0.483918
EMPLOYED_YEARS                     0.180072
GOODSPRICE_TO_AMT_CREDIT           0.000904
AMT_ANNUITY_TO_INCOME_RATIO        0.000039
AMT_ANNUITY_TO_CREDIT_RATIO        0.000039
INCOME_PER_FAMILY                  0.000007
YEARS_SINCE_PHONE_CHANGE           0.000003

Final shape: (307511, 139)


## 11. Save Output

In [16]:
application_train.to_parquet(PATH_OUTPUT, index=False)
print(f"Saved → {PATH_OUTPUT}")

Saved → a:\Dominic\My Projects\Home Credit\data\Model-Training\Staging\application_train_data_processed.parquet
